# QuACK demo

In [ ]:
from qick import *
import matplotlib.pyplot as plt
import numpy as np

import os
import time
from pynq import Overlay, DefaultIP, PL

We create a QickSoc object, but with our new firmware instead of the standard 216 bit file.

In [ ]:

#soc = QickSoc()
from qick.quack import QuackSoc
soc = QuackSoc(bitfile="../qick_lib/qick/quack_v2.bit", unipolar = False)
#soc = QuackSoc(bitfile="../qick_lib/qick/quackv1.bit", unipolar = False)
print(soc)
soccfg = QickConfig(soc.get_cfg())

ImportError: cannot import name 'QickSoc' from 'qick' (C:\Users\alwessels\Documents\Python\ats-qick\qick\qick_lib\qick\__init__.py)

Check that axi_pvp_gen_v7_0 is in the firmware we loaded (look towards the end of the list).

In [3]:
soc.ip_dict.keys()

dict_keys(['axi_dma_avg', 'axi_dma_buf', 'axi_dma_gen', 'axi_dma_mr', 'axi_dma_tproc', 'axi_intc_0', 'axi_pvp_gen_v7_0', 'axis_pfb_readout_v4_0', 'axis_sg_mixmux8_v1_0', 'ddr4/axis_buffer_ddr_v1_0', 'axis_avg_buffer_0', 'axis_avg_buffer_1', 'axis_avg_buffer_2', 'axis_avg_buffer_3', 'axis_avg_buffer_4', 'axis_avg_buffer_5', 'axis_avg_buffer_6', 'axis_avg_buffer_7', 'axis_avg_buffer_8', 'axis_avg_buffer_9', 'axis_sg_int4_v2_0', 'axis_sg_int4_v2_1', 'axis_sg_int4_v2_10', 'axis_sg_int4_v2_2', 'axis_sg_int4_v2_3', 'axis_sg_int4_v2_4', 'axis_sg_int4_v2_5', 'axis_sg_int4_v2_6', 'axis_sg_int4_v2_7', 'axis_sg_int4_v2_8', 'axis_sg_int4_v2_9', 'axis_signal_gen_v6_0', 'axis_signal_gen_v6_1', 'axis_signal_gen_v6_2', 'axis_signal_gen_v6_3', 'axis_switch_avg', 'axis_switch_buf', 'axis_switch_ddr', 'axis_switch_gen', 'axis_switch_mr', 'mr_buffer_et_0', 'qick_processor_0', 'usp_rf_data_converter_0', 'zynq_ultra_ps_e_0'])

In [4]:
soc.ip_dict['axi_pvp_gen_v7_0']

{'fullpath': 'axi_pvp_gen_v7_0',
 'type': 'xilinx.com:module_ref:axi_pvp_gen_v7:1.0',
 'bdtype': None,
 'state': None,
 'addr_range': 4096,
 'phys_addr': 2684747776,
 'mem_id': 's_axi',
 'memtype': 'REGISTER',
 'gpio': {},
 'interrupts': {},
 'parameters': {'DATA_WIDTH': '32',
  'ADDR_WIDTH': '6',
  'Component_Name': 'd_1_axi_pvp_gen_v7_0_2',
  'EDK_IPTYPE': 'PERIPHERAL',
  'C_BASEADDR': '0xA0060000',
  'C_HIGHADDR': '0xA0060FFF'},
 'registers': {},
 'device': <pynq.pl_server.embedded_device.EmbeddedDevice at 0xffff828ad100>,
 'driver': qick.drivers.peripherals.AxiPvpGen}

## Demo the DAC powers
These next few cells are to demonstrate how to initialize and set any given DAC. We only need one instance of the DAC class (we'll call it `bias`) to run the commands for all of the DACs.

Make sure you have a DAC plugged into slot 2! Or change the demux argument in the code. It's also a good idea to have a multimeter at hand here to check your output.
Note that any time a DAC is power cycled, you will need to rerun the init_DAC for it.

In [ ]:
soc.init_DAC(channel = 4)
soc.init_DAC(channel = 0)
soc.init_DAC(channel = 2)
soc.init_DAC(channel = 6)
soc.set_DAC(4,0)
soc.set_DAC(0,0)
soc.set_DAC(2,0)
soc.set_DAC(6,0)

## GvG with qickquack DACs
This is a qick program that in the past would have triggered an external PXI unit via the PMOD0_0 pin. Now that is internally connected to output 7, so we control all the settings for this experiment from this notebook.

In [ ]:
soc.clear_dac_regs()       
class Gvg(asm_v2.AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        self.declare_gen(ch=gen_ch, nqz=cfg['nqz'])
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])
       
        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['freq'], gen_ch=gen_ch,
                              outsel='product'
                              )
        self.add_pulse(ch=gen_ch, name="sourcedrain", ro_ch=ro_ch,
                       style="const",
                       freq=cfg['freq'],
                       length=cfg['pulse_len'],
                       phase=0,
                       phrst = 1,
                       gain=cfg['gain'],
                      ) #<-pulse length is most of the dwell time
        self.send_readoutconfig(ch=cfg['ro_ch'], name="myro", t=0)
       
        ### setup QuACK dacs here! ###
        self.add_loop("myloop3", 2)
        self.add_loop("myloop2", 2)
        self.add_loop("myloop1", 4)

        soc.setup_quack_sweep(loop_0_cfg = {'start': [0,0,0,0], 'step': [0.5, 0.5, 0.5, 0.5], 'channels': [2], 'loop_size':4},
                               loop_1_cfg = {'start': [0,0,0,0], 'step': [0.5, 0.5, 0.5, 0.5], 'channels': [6], 'loop_size':2},
                              loop_2_cfg = {'start': [0,0,0,0], 'step': [0.5, 0.5, 0.5, 0.5], 'channels': [4], 'loop_size':2},
                              dwell_cycles = 100*(cfg['pulse_len'] + cfg['meas_delay']))
        
        # go back to waiting for qick to trigger steps
        soc.axi_pvp_gen_v7_0.set_trigger_source('qick')
        self.trigger(pins=[7], t=0)
        self.delay_auto(t=cfg['meas_delay'] + cfg['pulse_len'])

       
    def _body(self, cfg):

        self.pulse(ch=cfg['gen_ch'], name="sourcedrain", t=cfg['meas_delay'])
        self.trigger(ros=[cfg['ro_ch']], t=cfg['meas_delay'], pins=[7])
        self.delay_auto(cfg['meas_delay'])
 
 
config = {
          'gen_ch': 0,
          'ro_ch': 0,
          'freq': 10,
          'nqz': 1,
          'trig_time': 0,
          'ro_len': 10,
          'pulse_len': 20,
          'gain': 0.5,
          'meas_delay': 10,
          }



# the 7th trigger disappears when it gets connected to the pvp gen block, so add it back in if it's missing
pin_list = soccfg['tprocs'][0]['output_pins'] 
if (len(pin_list) < 8):
    pin_list.append(('trig', 7, 0, 'trigger_pmod'))                     
gvgprog = Gvg(soccfg, reps=1, final_delay=0, cfg=config) #make plot square by setting all dims to same width



1
initializing dac 0
setting dac 0 to 0
initializing dac 2
setting dac 2 to 0
initializing dac 4
setting dac 4 to 0
initializing dac 6
setting dac 6 to 0
step size is 2097.000000
step size is 2097.000000
step size is 2097.000000
step size is 2097.000000


In [ ]:
d = gvgprog.acquire(soc)


  0%|          | 0/100 [00:00<?, ?it/s]

If all has gone well and your oscope was set up correctly, you should see some beautifully stepped DAC outputs, with readouts that line up!

In [ ]:
soc.axi_pvp_gen_v7_0.quit_pvp()